## 1. 📦 Installation des dépendances

In [113]:
!pip install -q sentence-transformers faiss-cpu

## 2. 📊 Chargement et exploration du dataset

In [114]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Chargement du dataset réel
df = pd.read_csv("/kaggle/input/datasets/elkorachiinsaf/datacsvv/data.csv")

print("=" * 60)
print("📊 INFORMATIONS SUR LE DATASET")
print("=" * 60)
print(f"Nombre de lignes      : {len(df)}")
print(f"Nombre de colonnes    : {len(df.columns)}")
print(f"Colonnes              : {list(df.columns)}")
print()
print("Types de paragraphes  :", df['type_paragraphe'].value_counts().to_dict())
print("Catégories véhicules  :", df['categorie_vehicule'].value_counts().to_dict())
print()
df.head()

📊 INFORMATIONS SUR LE DATASET
Nombre de lignes      : 6
Nombre de colonnes    : 7
Colonnes              : ['article_id', 'infraction_desc', 'categorie_vehicule', 'amende_fixe', 'points_retrait', 'mots_cles', 'type_paragraphe']

Types de paragraphes  : {'autorisation': 3, 'autre': 2, 'obligation': 1}
Catégories véhicules  : {'general': 3, 'militaire': 1, 'specialise': 1, 'moto': 1}



,article_id,infraction_desc,categorie_vehicule,amende_fixe,points_retrait,mots_cles,type_paragraphe
0,2,الوطني خالل مده صالحيه الرخصه املذكوره دون ان ...,general,NaN,NaN,رخصه,autre
1,3,يجب علي الساءقين الحاصلين علي رخصه سياقه مسلمه...,general,NaN,NaN,"سياقه, رخصه",obligation
2,4,في حاله السير الدولي ووفقا لالتفاقيه الدوليه ل...,general,NaN,NaN,"سياقه, رخصه, دولي",autorisation
3,5,من 13 بتاريخ 1.16. رقم الشريف 012ال 34567بت ر8...,militaire,NaN,NaN,"سياقه, رخصه, طريق, عسكري",autorisation
4,6,ال يجوز الي كان سياقه مركبه فالحيه ذات محرك او...,specialise,NaN,NaN,"سياقه, رخصه, مركبه, طريق",autorisation


## 3. 🧹 Prétraitement et nettoyage des textes

In [115]:
import re

def clean_arabic_text(text):
    """
    Nettoie le texte arabe :
    - Supprime les retours à la ligne
    - Normalise les espaces multiples
    - Supprime les caractères spéciaux inutiles
    """
    text = str(text)
    text = text.lower()                           # Minuscules (utile pour latin mélangé)
    text = re.sub(r'\n+', ' ', text)              # Remplace les sauts de ligne
    text = re.sub(r'\s+', ' ', text)              # Normalise les espaces
    text = text.strip()
    return text

# Application du nettoyage
df['infraction_desc'] = df['infraction_desc'].astype(str)
df['texte'] = df['infraction_desc'].apply(clean_arabic_text)

# Suppression des lignes vides
df = df[df['texte'].str.len() > 10].reset_index(drop=True)

print(f"✅ Nombre de documents après nettoyage : {len(df)}")
print()
print("Exemple de texte nettoyé :")
print("-" * 60)
print(df['texte'].iloc[0][:300])
df[['article_id', 'texte', 'type_paragraphe', 'categorie_vehicule']].head()

✅ Nombre de documents après nettoyage : 6

Exemple de texte nettoyé :
------------------------------------------------------------
الوطني خالل مده صالحيه الرخصه املذكوره دون ان تتجاوز املده املشار اليها في البند .اعاله


,article_id,texte,type_paragraphe,categorie_vehicule
0,2,الوطني خالل مده صالحيه الرخصه املذكوره دون ان ...,autre,general
1,3,يجب علي الساءقين الحاصلين علي رخصه سياقه مسلمه...,obligation,general
2,4,في حاله السير الدولي ووفقا لالتفاقيه الدوليه ل...,autorisation,general
3,5,من 13 بتاريخ 1.16. رقم الشريف 012ال 34567بت ر8...,autorisation,militaire
4,6,ال يجوز الي كان سياقه مركبه فالحيه ذات محرك او...,autorisation,specialise


## 4. ✂️ Chunking intelligent des documents

In [116]:
def split_text_with_overlap(text, chunk_size=80, overlap=15):
    """
    Découpe le texte en chunks avec chevauchement pour préserver le contexte.
    
    Args:
        text      : texte à découper
        chunk_size: nombre de mots par chunk
        overlap   : chevauchement entre chunks consécutifs (en mots)
    
    Returns:
        Liste de chunks textuels
    """
    words = str(text).split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap  # Avancement avec chevauchement
    
    return chunks

# Construction de la base de connaissance
chunks = []
chunk_metadata = []  # Métadonnées associées à chaque chunk

for _, row in df.iterrows():
    text_chunks = split_text_with_overlap(row['texte'], chunk_size=80, overlap=15)
    for ch in text_chunks:
        if ch.strip():
            chunks.append(ch)
            chunk_metadata.append({
                'article_id'        : row['article_id'],
                'type_paragraphe'   : row['type_paragraphe'],
                'categorie_vehicule': row['categorie_vehicule'],
                'mots_cles'         : row.get('mots_cles', '')
            })

print(f"✅ Nombre de chunks créés    : {len(chunks)}")
print(f"   Chunk size moyen (mots)   : {np.mean([len(c.split()) for c in chunks]):.1f}")
print(f"   Articles couverts         : {len(set(m['article_id'] for m in chunk_metadata))}")
print()
print("Exemple de chunk :")
print("-" * 60)
print(chunks[0])

✅ Nombre de chunks créés    : 6
   Chunk size moyen (mots)   : 42.7
   Articles couverts         : 6

Exemple de chunk :
------------------------------------------------------------
الوطني خالل مده صالحيه الرخصه املذكوره دون ان تتجاوز املده املشار اليها في البند .اعاله


In [117]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(torch.cuda.is_available())

True


In [118]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Modèle chargé avec succès !")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle chargé avec succès !


In [119]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cpu"
)

print("✅ Modèle d'embedding chargé sur CPU avec succès !")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle d'embedding chargé sur CPU avec succès !


## 5. 🧠 Modèle d'embedding multilingue

> **Choix clé :** `paraphrase-multilingual-MiniLM-L12-v2` supporte **50+ langues** dont l'arabe,
> contrairement à `all-MiniLM-L6-v2` qui est anglais uniquement.

In [120]:
from sentence_transformers import SentenceTransformer

print("⏳ Chargement du modèle d'embedding multilingue...")
print("   Modèle : paraphrase-multilingual-MiniLM-L12-v2")
print("   Langues supportées : 50+ (arabe, français, anglais, ...)")
print()

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("✅ Modèle chargé avec succès !")
print(f"   Dimension des vecteurs : {embedding_model.get_sentence_embedding_dimension()}")

⏳ Chargement du modèle d'embedding multilingue...
   Modèle : paraphrase-multilingual-MiniLM-L12-v2
   Langues supportées : 50+ (arabe, français, anglais, ...)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle chargé avec succès !
   Dimension des vecteurs : 384


## 6. 🗄️ Construction de l'index FAISS

In [121]:
import faiss

print("⏳ Génération des embeddings pour tous les chunks...")
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True  # Normalisation L2 → meilleure similarité cosinus
)
embeddings = np.array(embeddings).astype("float32")

# Construction de l'index FAISS
dimension = embeddings.shape[1]

# IndexFlatIP = produit scalaire (équivalent cosinus si embeddings normalisés)
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print()
print("=" * 50)
print(f"✅ Index FAISS construit !")
print(f"   Dimension des vecteurs   : {dimension}")
print(f"   Nombre de vecteurs indexés : {index.ntotal}")
print(f"   Métrique utilisée        : Similarité cosinus (IP normalisé)")
print("=" * 50)

⏳ Génération des embeddings pour tous les chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Index FAISS construit !
   Dimension des vecteurs   : 384
   Nombre de vecteurs indexés : 6
   Métrique utilisée        : Similarité cosinus (IP normalisé)


## 7. 🔍 Fonction de retrieval sémantique

In [122]:
def retrieve(query, k=3, verbose=False):
    """
    Recherche les k chunks les plus pertinents pour une requête donnée.
    
    Args:
        query   : question de l'utilisateur (arabe ou français)
        k       : nombre de documents à retourner
        verbose : affiche les détails si True
    
    Returns:
        Liste de dicts {chunk, article_id, score, metadata}
    """
    # Encode la requête
    query_vec = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")
    
    # Recherche dans l'index FAISS
    scores, indices = index.search(query_vec, k)
    
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        result = {
            'rank'      : rank + 1,
            'chunk'     : chunks[idx],
            'score'     : float(score),
            'article_id': chunk_metadata[idx]['article_id'],
            'metadata'  : chunk_metadata[idx]
        }
        results.append(result)
        
        if verbose:
            print(f"  Rang {rank+1} | Article {result['article_id']} | Score: {score:.4f}")
            print(f"  Type: {chunk_metadata[idx]['type_paragraphe']}")
            print(f"  {result['chunk'][:150]}...")
            print()
    
    return results

# Test du retrieval
print("🔍 Test de la fonction de retrieval")
print("=" * 60)
print("Requête : ما هي عقوبة السياقة بدون رخصة؟")
print()
test_results = retrieve("ما هي عقوبة السياقة بدون رخصة؟", k=3, verbose=True)

🔍 Test de la fonction de retrieval
Requête : ما هي عقوبة السياقة بدون رخصة؟

  Rang 1 | Article 3 | Score: 0.2875
  Type: obligation
  يجب علي الساءقين الحاصلين علي رخصه سياقه مسلمه بالخارج، بعد انصرام املده املشار اليها في املاده السابقه، ان يتقدموا المتحانات الحصول علي رخصه السياقه ...

  Rang 2 | Article 4 | Score: 0.1585
  Type: autorisation
  في حاله السير الدولي ووفقا لالتفاقيه الدوليه للسير علي الطرق، تسلم الهيءات املءهله لذلك من .قبل االداره، رخصه دوليه للسياقه موضوعه في دفتر خاص يجوز لل...

  Rang 3 | Article 6 | Score: 0.1194
  Type: autorisation
  ال يجوز الي كان سياقه مركبه فالحيه ذات محرك او مركبه غابويه ذات محرك او اريبه لالشغال العموميه او اريبه خاصه ذات محرك، علي الطريق العموميه، مالم يكن ح...



## 8. 🤖 LLM 

In [123]:
!pip install -q transformers accelerate bitsandbytes sentencepiece openai

In [124]:
from transformers import AutoTokenizer
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
import torch

In [125]:
# =========================================================
# INSTALLATION
# =========================================================
!pip install -q transformers accelerate bitsandbytes sentencepiece openai

# =========================================================
# IMPORTS
# =========================================================
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# =========================================================
# NETTOYAGE MEMOIRE
# =========================================================
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

clear_memory()

# =========================================================
# CONFIG QUANTIZATION
# =========================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# =========================================================
# NOMS DES MODELES
# =========================================================
QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MISTRAL_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
GPT_MODEL_NAME = "gpt-4o-mini"

# =========================================================
# CHARGEMENT GPT (API)
# =========================================================
user_secrets = UserSecretsClient()
OPENAI_API_KEY = user_secrets.get_secret("OPENAI_API_KEY")
gpt_client = OpenAI(api_key=OPENAI_API_KEY)

# =========================================================
# VARIABLES GLOBALES
# =========================================================
qwen_tokenizer = None
qwen_model = None

mistral_tokenizer = None
mistral_model = None

# =========================================================
# DECHARGER MODELES LOCAUX
# =========================================================
def unload_local_models():
    global qwen_tokenizer, qwen_model, mistral_tokenizer, mistral_model

    try:
        del qwen_model
    except:
        pass

    try:
        del qwen_tokenizer
    except:
        pass

    try:
        del mistral_model
    except:
        pass

    try:
        del mistral_tokenizer
    except:
        pass

    qwen_model = None
    qwen_tokenizer = None
    mistral_model = None
    mistral_tokenizer = None

    clear_memory()

# =========================================================
# CHARGEMENT QWEN
# =========================================================
def load_qwen():
    tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
    return tokenizer, model

# =========================================================
# CHARGEMENT MISTRAL LOCAL LEGER
# =========================================================
def load_mistral():
    tokenizer = AutoTokenizer.from_pretrained(MISTRAL_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MISTRAL_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
    return tokenizer, model

# =========================================================
# FORMAT PROMPT
# =========================================================
def build_full_prompt(prompt, system_prompt=None):
    if system_prompt:
        return f"{system_prompt}\n\n{prompt}"
    return prompt

# =========================================================
# APPEL QWEN
# =========================================================
def call_qwen(prompt, system_prompt=None, max_new_tokens=128):
    global qwen_tokenizer, qwen_model

    if qwen_model is None or qwen_tokenizer is None:
        unload_local_models()
        qwen_tokenizer, qwen_model = load_qwen()

    full_prompt = build_full_prompt(prompt, system_prompt)

    inputs = qwen_tokenizer(
        full_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(qwen_model.device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    text = qwen_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text

# =========================================================
# APPEL MISTRAL LOCAL LEGER
# =========================================================
def call_mistral(prompt, system_prompt=None, max_new_tokens=128):
    global mistral_tokenizer, mistral_model

    if mistral_model is None or mistral_tokenizer is None:
        unload_local_models()
        mistral_tokenizer, mistral_model = load_mistral()

    full_prompt = build_full_prompt(prompt, system_prompt)

    inputs = mistral_tokenizer(
        full_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(mistral_model.device)

    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    text = mistral_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text

# =========================================================
# APPEL GPT
# =========================================================
def call_gpt(prompt, system_prompt=None, max_new_tokens=128):
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": prompt})

    response = gpt_client.chat.completions.create(
        model=GPT_MODEL_NAME,
        messages=messages,
        max_tokens=max_new_tokens,
        temperature=0.2
    )

    return response.choices[0].message.content

# =========================================================
# FONCTION GENERALE
# =========================================================
def call_llm(prompt, system_prompt=None, max_new_tokens=128, model_name="qwen"):
    if model_name == "qwen":
        return call_qwen(
            prompt=prompt,
            system_prompt=system_prompt,
            max_new_tokens=max_new_tokens
        )

    elif model_name == "mistral":
        return call_mistral(
            prompt=prompt,
            system_prompt=system_prompt,
            max_new_tokens=max_new_tokens
        )

    elif model_name == "gpt":
        return call_gpt(
            prompt=prompt,
            system_prompt=system_prompt,
            max_new_tokens=max_new_tokens
        )

    else:
        raise ValueError("model_name doit être : 'qwen', 'mistral' ou 'gpt'")

# =========================================================
# TEST SIMPLE
# =========================================================
SYSTEM_PROMPT_TEST = "أنت مساعد قانوني متخصص في قانون السير المغربي."
prompt_test = "ما هي عقوبة السياقة بدون رخصة؟"

print("===== TEST GPT =====")
print(call_llm(
    prompt=prompt_test,
    system_prompt=SYSTEM_PROMPT_TEST,
    model_name="gpt",
    max_new_tokens=100
))

print("\n===== TEST QWEN =====")
print(call_llm(
    prompt=prompt_test,
    system_prompt=SYSTEM_PROMPT_TEST,
    model_name="qwen",
    max_new_tokens=100
))

print("\n===== TEST MISTRAL =====")
print(call_llm(
    prompt=prompt_test,
    system_prompt=SYSTEM_PROMPT_TEST,
    model_name="mistral",
    max_new_tokens=100
))

===== TEST GPT =====
في المغرب، تعتبر السياقة بدون رخصة مخالفة قانونية خطيرة. وفقًا لقانون السير، فإن عقوبة السياقة بدون رخصة قد تشمل:

1. **غرامة مالية**: يمكن أن تتراوح الغرامة بين 300 و 1,200 درهم.
2. **الحبس**: في بعض الحالات، قد تصل العقوبة إلى الحبس لمدة قد تصل إلى 6 أشهر.
3. **سحب السيارة**: يمكن أن يتم س

===== TEST QWEN =====


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

أنت مساعد قانوني متخصص في قانون السير المغربي.

ما هي عقوبة السياقة بدون رخصة؟
العقوبات المحتملة للسائق الذي يسيء إلى سرعة التسجيل أو الاتصال بالرقم 1300، أو أي من أشكال السرعة غير المرغوب فيها، أو أي من أشكال السرعة التي تقلل من الرقابة على السائق، أو أي من أشكال السرعة التي تقلل من الرقابة على السائق، أو أي من أشكال السر

===== TEST MISTRAL =====


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

أنت مساعد قانوني متخصص في قانون السير المغربي.

ما هي عقوبة السياقة بدون رخصة؟ 

السياقة هي من خلف من أجل تحقيق معلومات متعددة ومتكاملة من خلف من أجل تحقيق معلومات متعددة ومتكاملة


In [126]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [127]:
def load_qwen():
    clear_gpu()
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )
    return tokenizer, model

def call_qwen(prompt, system_prompt=None, max_new_tokens=256):
    tokenizer, model = load_qwen()

    full_prompt = f"{system_prompt}\n\n{prompt}" if system_prompt else prompt

    inputs = tokenizer(
        full_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=768
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    del model, tokenizer, inputs, outputs
    clear_gpu()

    return text

In [128]:
def load_mistral():
    import gc, torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16
    )

    return tokenizer, model


def call_mistral(prompt, system_prompt=None, max_new_tokens=256):
    tokenizer, model = load_mistral()

    full_prompt = f"{system_prompt}\n\n{prompt}" if system_prompt else prompt

    inputs = tokenizer(
        full_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # nettoyage mémoire
    del model, tokenizer, inputs, outputs
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()

    return text

In [129]:
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
OPENAI_API_KEY = user_secrets.get_secret("OPENAI_API_KEY")
gpt_client = OpenAI(api_key=OPENAI_API_KEY)

def call_gpt(prompt, system_prompt=None, max_new_tokens=256):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    response = gpt_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        max_tokens=max_new_tokens,
        temperature=0.2
    )
    return response.choices[0].message.content

In [130]:
def call_llm(prompt, system_prompt=None, max_new_tokens=256, model_name="qwen"):
    if model_name == "qwen":
        return call_qwen(prompt, system_prompt=system_prompt, max_new_tokens=max_new_tokens)
    elif model_name == "mistral":
        return call_mistral(prompt, system_prompt=system_prompt, max_new_tokens=max_new_tokens)
    elif model_name == "gpt":
        return call_gpt(prompt, system_prompt=system_prompt, max_new_tokens=max_new_tokens)
    else:
        raise ValueError("model_name doit être : qwen, mistral, gpt")

## 9. 🔗 Pipeline RAG complet

In [131]:
SYSTEM_PROMPT = """أنت مساعد قانوني متخصص في قانون السير والجولان المغربي.
مهمتك هي الإجابة على أسئلة المستخدمين بدقة وبشكل واضح،
مستنداً فقط على المقاطع القانونية المقدمة إليك كسياق.

قواعد مهمة:
- أجب دائماً بالعربية إذا كان السؤال بالعربية
- استند فقط على المعلومات الواردة في السياق المقدم
- إذا لم تجد المعلومة في السياق، قل ذلك بوضوح
- اذكر رقم المادة المصدر في إجابتك
- كن موجزاً ودقيقاً"""

def rag_answer(query, k=2, llm_name="qwen", verbose=False):
    if verbose:
        print("🔍 Étape 1 : Retrieval...")

    retrieved_docs = retrieve(query, k=k, verbose=verbose)

    if not retrieved_docs:
        return {
            "query": query,
            "answer": "لم يتم العثور على مواد قانونية مناسبة للإجابة على هذا السؤال.",
            "sources": [],
            "context": ""
        }

    if verbose:
        print("📝 Étape 2 : Construction du contexte...")

    context_parts = []
    for doc in retrieved_docs:
        context_parts.append(
            f"[المادة {doc['article_id']} - {doc['metadata']['type_paragraphe']}]\n{doc['chunk']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""استناداً فقط على المقاطع القانونية التالية من قانون السير المغربي:

=== السياق القانوني ===
{context}
======================

السؤال: {query}

أجب بشكل واضح وموجز، مع ذكر أرقام المواد القانونية المستعملة.
إذا لم تكن الإجابة موجودة بوضوح في السياق، فقل ذلك صراحة.
"""

    if verbose:
        print(f"🤖 Étape 3 : Génération avec {llm_name.upper()}...")

    answer = call_llm(
        prompt=prompt,
        system_prompt=SYSTEM_PROMPT,
        max_new_tokens=128,
        model_name=llm_name
    )

    return {
        "query": query,
        "answer": answer,
        "sources": retrieved_docs,
        "context": context
    }

In [132]:
result = rag_answer(
    "ما هي عقوبة السياقة بدون رخصة؟",
    k=2,
    llm_name="gpt",
    verbose=True
)

print(result["answer"])

🔍 Étape 1 : Retrieval...
  Rang 1 | Article 3 | Score: 0.2875
  Type: obligation
  يجب علي الساءقين الحاصلين علي رخصه سياقه مسلمه بالخارج، بعد انصرام املده املشار اليها في املاده السابقه، ان يتقدموا المتحانات الحصول علي رخصه السياقه ...

  Rang 2 | Article 4 | Score: 0.1585
  Type: autorisation
  في حاله السير الدولي ووفقا لالتفاقيه الدوليه للسير علي الطرق، تسلم الهيءات املءهله لذلك من .قبل االداره، رخصه دوليه للسياقه موضوعه في دفتر خاص يجوز لل...

📝 Étape 2 : Construction du contexte...
🤖 Étape 3 : Génération avec GPT...
لا توجد معلومات واضحة في السياق القانوني المقدم حول عقوبة السياقة بدون رخصة. لذلك، لا أستطيع تقديم إجابة دقيقة عن هذا السؤال.


## 10. 🧪 Tests du pipeline RAG

In [133]:
def display_rag_result(result):
    """Affiche le résultat RAG de façon lisible."""
    print("=" * 70)
    print(f"❓ QUESTION : {result['query']}")
    print("=" * 70)
    print()
    print("📚 DOCUMENTS RÉCUPÉRÉS :")
    for doc in result['sources']:
        print(f"  • Article {doc['article_id']} (score: {doc['score']:.4f}) — {doc['metadata']['type_paragraphe']}")
        print(f"    {doc['chunk'][:120]}...")
    print()
    print("🤖 RÉPONSE GÉNÉRÉE PAR LE LLM :")
    print("-" * 70)
    print(result['answer'])
    print("=" * 70)
    print()

# ── Test 1 : Question sur la conduite sans permis ─────────────────────
print("TEST 1")
result1 = rag_answer("ما هي عقوبة السياقة بدون رخصة؟", k=3)
display_rag_result(result1)

TEST 1


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

❓ QUESTION : ما هي عقوبة السياقة بدون رخصة؟

📚 DOCUMENTS RÉCUPÉRÉS :
  • Article 3 (score: 0.2875) — obligation
    يجب علي الساءقين الحاصلين علي رخصه سياقه مسلمه بالخارج، بعد انصرام املده املشار اليها في املاده السابقه، ان يتقدموا المت...
  • Article 4 (score: 0.1585) — autorisation
    في حاله السير الدولي ووفقا لالتفاقيه الدوليه للسير علي الطرق، تسلم الهيءات املءهله لذلك من .قبل االداره، رخصه دوليه للسي...
  • Article 6 (score: 0.1194) — autorisation
    ال يجوز الي كان سياقه مركبه فالحيه ذات محرك او مركبه غابويه ذات محرك او اريبه لالشغال العموميه او اريبه خاصه ذات محرك، ع...

🤖 RÉPONSE GÉNÉRÉE PAR LE LLM :
----------------------------------------------------------------------
أنت مساعد قانوني متخصص في قانون السير والجولان المغربي.
مهمتك هي الإجابة على أسئلة المستخدمين بدقة وبشكل واضح،
مستنداً فقط على المقاطع القانونية المقدمة إليك كسياق.

قواعد مهمة:
- أجب دائماً بالعربية إذا كان السؤال بالعربية
- استند فقط على المعلومات الواردة في السياق المقدم
- إذا لم تجد المعلومة في السياق، قل 

In [134]:
# ── Test 2 : Question sur le permis international ─────────────────────
print("TEST 2")
result2 = rag_answer("هل يمكن للأجانب السياقة في المغرب برخصتهم الأجنبية؟", k=3)
display_rag_result(result2)

TEST 2


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

❓ QUESTION : هل يمكن للأجانب السياقة في المغرب برخصتهم الأجنبية؟

📚 DOCUMENTS RÉCUPÉRÉS :
  • Article 3 (score: 0.5025) — obligation
    يجب علي الساءقين الحاصلين علي رخصه سياقه مسلمه بالخارج، بعد انصرام املده املشار اليها في املاده السابقه، ان يتقدموا المت...
  • Article 4 (score: 0.2682) — autorisation
    في حاله السير الدولي ووفقا لالتفاقيه الدوليه للسير علي الطرق، تسلم الهيءات املءهله لذلك من .قبل االداره، رخصه دوليه للسي...
  • Article 6 (score: 0.2444) — autorisation
    ال يجوز الي كان سياقه مركبه فالحيه ذات محرك او مركبه غابويه ذات محرك او اريبه لالشغال العموميه او اريبه خاصه ذات محرك، ع...

🤖 RÉPONSE GÉNÉRÉE PAR LE LLM :
----------------------------------------------------------------------
أنت مساعد قانوني متخصص في قانون السير والجولان المغربي.
مهمتك هي الإجابة على أسئلة المستخدمين بدقة وبشكل واضح،
مستنداً فقط على المقاطع القانونية المقدمة إليك كسياق.

قواعد مهمة:
- أجب دائماً بالعربية إذا كان السؤال بالعربية
- استند فقط على المعلومات الواردة في السياق المقدم
- إذا لم تجد ال

In [135]:
# ── Test 3 : Question en français ─────────────────────────────────────
print("TEST 3 (question en français)")
result3 = rag_answer("Quelles sont les obligations des conducteurs étrangers au Maroc ?", k=3)
display_rag_result(result3)

TEST 3 (question en français)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

❓ QUESTION : Quelles sont les obligations des conducteurs étrangers au Maroc ?

📚 DOCUMENTS RÉCUPÉRÉS :
  • Article 3 (score: 0.4340) — obligation
    يجب علي الساءقين الحاصلين علي رخصه سياقه مسلمه بالخارج، بعد انصرام املده املشار اليها في املاده السابقه، ان يتقدموا المت...
  • Article 4 (score: 0.2428) — autorisation
    في حاله السير الدولي ووفقا لالتفاقيه الدوليه للسير علي الطرق، تسلم الهيءات املءهله لذلك من .قبل االداره، رخصه دوليه للسي...
  • Article 6 (score: 0.2236) — autorisation
    ال يجوز الي كان سياقه مركبه فالحيه ذات محرك او مركبه غابويه ذات محرك او اريبه لالشغال العموميه او اريبه خاصه ذات محرك، ع...

🤖 RÉPONSE GÉNÉRÉE PAR LE LLM :
----------------------------------------------------------------------
أنت مساعد قانوني متخصص في قانون السير والجولان المغربي.
مهمتك هي الإجابة على أسئلة المستخدمين بدقة وبشكل واضح،
مستنداً فقط على المقاطع القانونية المقدمة إليك كسياق.

قواعد مهمة:
- أجب دائماً بالعربية إذا كان السؤال بالعربية
- استند فقط على المعلومات الواردة في السياق المقدم
-

## 11. 📊 Évaluation du pipeline RAG

> Comparaison **avec RAG** vs **sans RAG** pour montrer l'apport du retrieval.

In [136]:
def generate_rag_answer(query, k=3, threshold=0.25):
    if is_out_of_domain(query):
        return "❌ Cette question est hors domaine. Le système répond seulement aux questions liées au code de la route."

    results = search(query, k=k)

    if not results:
        return "❌ Aucun texte juridique pertinent n'a été trouvé."

    # Détection hors domaine par score
    if results[0]["score"] < threshold:
        return "❌ Question probablement hors domaine ou information insuffisante dans la base."

    refs = ", ".join([str(r['article_id']) for r in results])
    context = "\n\n".join(
        [f"Article {r['article_id']} : {r['chunk']}" for r in results]
    )

    prompt = f"""
Tu es un assistant juridique spécialisé dans le code de la route marocain.

Réponds uniquement à partir du contexte ci-dessous.
Si l'information n'est pas claire dans le contexte, dis-le.
Réponds de manière courte, claire et structurée.
Cite les articles utilisés à la fin.

Contexte :
{context}

Question :
{query}

Réponse :
"""

    llm_answer = call_llm(prompt)

    final_answer = f"""{llm_answer}

Articles de référence : {refs}
"""
    return final_answer

In [137]:
# Métriques quantitatives du retrieval
print("📈 MÉTRIQUES DU PIPELINE")
print("=" * 50)

test_queries = [
    "ما هي عقوبة السياقة بدون رخصة؟",
    "هل يمكن للأجانب السياقة في المغرب؟",
    "ما هي أصناف رخصة السياقة؟",
]

scores_list = []
for query in test_queries:
    docs = retrieve(query, k=3)
    avg_score = np.mean([d['score'] for d in docs])
    top_score = docs[0]['score']
    scores_list.append({'query': query, 'top_score': top_score, 'avg_score': avg_score})
    print(f"Requête : {query[:50]}")
    print(f"  → Top-1 score : {top_score:.4f} | Moyenne top-3 : {avg_score:.4f}")
    print()

overall_top = np.mean([s['top_score'] for s in scores_list])
overall_avg = np.mean([s['avg_score'] for s in scores_list])
print("-" * 50)
print(f"📊 Score moyen Top-1   : {overall_top:.4f}")
print(f"📊 Score moyen Top-3   : {overall_avg:.4f}")
print()
print("✅ Score > 0.5 = bonne similarité sémantique")
print("✅ Modèle multilingue → pertinence améliorée vs all-MiniLM-L6-v2")

📈 MÉTRIQUES DU PIPELINE
Requête : ما هي عقوبة السياقة بدون رخصة؟
  → Top-1 score : 0.2875 | Moyenne top-3 : 0.1885

Requête : هل يمكن للأجانب السياقة في المغرب؟
  → Top-1 score : 0.4888 | Moyenne top-3 : 0.3548

Requête : ما هي أصناف رخصة السياقة؟
  → Top-1 score : 0.3022 | Moyenne top-3 : 0.1505

--------------------------------------------------
📊 Score moyen Top-1   : 0.3595
📊 Score moyen Top-3   : 0.2312

✅ Score > 0.5 = bonne similarité sémantique
✅ Modèle multilingue → pertinence améliorée vs all-MiniLM-L6-v2


## 12. 💬 Interface Chat Interactive

In [138]:
class RAGChatbot:
    """
    Chatbot RAG avec mémoire de conversation.
    Maintient l'historique des échanges pour un dialogue cohérent.
    """

    def __init__(self):
        self.history = []  # Historique de la conversation
        self.session_count = 0

    def chat(self, query, k=3, llm_name="gpt"):
        """Traite une question et retourne la réponse avec historique."""
        self.session_count += 1

        # Retrieval des documents pertinents
        retrieved_docs = retrieve(query, k=k)

        if not retrieved_docs:
            answer = "لم يتم العثور على مواد قانونية مناسبة للإجابة على هذا السؤال."
            self.history.append((query, answer))
            return {
                "answer": answer,
                "sources": []
            }

        context_parts = []
        for doc in retrieved_docs:
            context_parts.append(
                f"[المادة {doc['article_id']}]\n{doc['chunk']}"
            )
        context = "\n\n".join(context_parts)

        # Historique sous forme texte
        history_text = ""
        for past_q, past_a in self.history[-4:]:
            history_text += f"سؤال سابق: {past_q}\n"
            history_text += f"جواب سابق: {past_a}\n\n"

        # Prompt actuel avec contexte + historique
        current_prompt = f"""السياق القانوني:
{context}

سجل المحادثة السابق:
{history_text}

السؤال الحالي:
{query}

أجب بشكل واضح وموجز، مع ذكر المواد القانونية المستعملة.
إذا لم تكن الإجابة موجودة بوضوح في السياق، فقل ذلك صراحة.
"""

        # Appel au LLM
        answer = call_llm(
            prompt=current_prompt,
            system_prompt=SYSTEM_PROMPT,
            model_name=llm_name,
            max_new_tokens=128
        )

        # Mise à jour de l'historique
        self.history.append((query, answer))

        return {
            "answer": answer,
            "sources": retrieved_docs
        }

    def reset(self):
        """Réinitialise la conversation."""
        self.history = []
        self.session_count = 0
        print("🔄 Conversation réinitialisée.")


# Initialisation du chatbot
chatbot = RAGChatbot()

print("✅ Chatbot RAG initialisé avec mémoire de conversation")
print()

# Démonstration du chat multi-tours
print("💬 DÉMONSTRATION : Conversation multi-tours")
print("=" * 70)

questions_demo = [
    "ما هي شروط الحصول على رخصة السياقة في المغرب؟",
    "وماذا عن الأجانب؟ هل لهم نفس الشروط؟",
]

for q in questions_demo:
    print(f"\n👤 UTILISATEUR : {q}")
    result = chatbot.chat(q, k=2, llm_name="gpt")
    print("\n🤖 ASSISTANT   :")
    print(result["answer"])
    print(f"\n📚 Sources     : Articles {[d['article_id'] for d in result['sources']]}")
    print("-" * 70)

✅ Chatbot RAG initialisé avec mémoire de conversation

💬 DÉMONSTRATION : Conversation multi-tours

👤 UTILISATEUR : ما هي شروط الحصول على رخصة السياقة في المغرب؟

🤖 ASSISTANT   :
شروط الحصول على رخصة السياقة في المغرب تتضمن ما يلي:

1. يجب على السائقين الحاصلين على رخصة سياقة مسلمة بالخارج، بعد انصرام المدة المحددة، أن يتقدموا للاختبارات للحصول على رخصة السياقة المغربية أو أن يطلبوا تبديل رخصتهم (المادة 3).

2. في حالة السير الدولي، يمكن للسائقين من جنسية أجنبية الحاصلين على رخصة دولية للسياقة، السياقة على التراب المغربي (المادة 4).

لم يتم ذكر تفاصيل إضافية حول الشروط

📚 Sources     : Articles [3, 4]
----------------------------------------------------------------------

👤 UTILISATEUR : وماذا عن الأجانب؟ هل لهم نفس الشروط؟

🤖 ASSISTANT   :
الأجانب لديهم شروط مختلفة قليلاً. وفقاً للمادة 4، يمكن للسائقين من جنسية أجنبية الحاصلين على رخصة دولية للسياقة أن يقودوا على التراب المغربي. لكن لم يتم ذكر تفاصيل إضافية حول الشروط المحددة التي تنطبق عليهم. 

لذا، الإجابة ليست واضحة تماماً في السياق

In [ ]:
# Interface chat en boucle (fonctionne en local, désactivée sur Kaggle)
def chat_interface():
    """
    Interface chat interactive en ligne de commande.
    Commandes spéciales :
      - 'exit' ou 'quit' : quitter
      - 'reset'          : réinitialiser la conversation
      - 'history'        : afficher l'historique
    """
    bot = RAGChatbot()
    
    print("=" * 70)
    print("🚗  مساعد قانون السير المغربي — RAG Chatbot")
    print("=" * 70)
    print("اكتب سؤالك بالعربية أو الفرنسية.")
    print("الأوامر: 'exit' للخروج | 'reset' لإعادة التشغيل | 'history' للسجل")
    print()
    
    while True:
        try:
            query = input("❓ سؤالك : ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Au revoir !")
            break
        
        if not query:
            continue
        
        if query.lower() in ['exit', 'quit', 'خروج']:
            print("👋 شكراً على استخدامك للمساعد القانوني !")
            break
        
        elif query.lower() == 'reset':
            bot.reset()
            continue
        
        elif query.lower() == 'history':
            print(f"\n📜 السجل ({len(bot.history)} تبادل) :")
            for i, (q, a) in enumerate(bot.history, 1):
                print(f"  [{i}] {q[:60]}...")
            print()
            continue
        
        # Traitement de la question
        print("⏳ جاري البحث والإجابة...")
        try:
            result = bot.chat(query, k=3)
            print(f"\n🤖 الإجابة :")
            print("-" * 50)
            print(result['answer'])
            print(f"\n📚 المصادر : مواد {[d['article_id'] for d in result['sources']]}")
            print()
        except Exception as e:
            print(f"❌ خطأ : {e}")
            print("تأكد من صحة مفتاح API Groq")
        
        print("=" * 70)


# Exécution — fonctionne en local / Jupyter interactif
# Sur Kaggle en mode batch : utiliser les tests ci-dessus
try:
    chat_interface()
except Exception:
    print("ℹ️  Interface chat non disponible en mode batch.")
    print("   Utilisez directement : rag_answer('votre question') ou chatbot.chat('votre question')")

🚗  مساعد قانون السير المغربي — RAG Chatbot
اكتب سؤالك بالعربية أو الفرنسية.
الأوامر: 'exit' للخروج | 'reset' لإعادة التشغيل | 'history' للسجل



## Limites et perspectives

### Limites
- Le dataset utilisé reste limité en couverture juridique
- Le découpage en chunks peut parfois séparer des informations liées
- Le modèle utilisé dans le notebook reste un modèle léger
- L’évaluation a été réalisée sur un petit jeu de test

### Perspectives
- Tester plusieurs LLMs de manière expérimentale complète
- Enrichir la base avec davantage d’articles de loi
- Améliorer la détection hors domaine avec un classifieur dédié
- Ajouter une interface web plus avancée